# Project 3 — Movie Streaming Platform Analysis
**Level:** Beginner  |  **Intern:** Sohan Nandi (CSN-OC-2026-474)  |  **Domain:** Data Analysis  |  **October 2026**

## Objective & Business Questions
Analyze the Netflix titles catalog.

- Movie vs TV-show split and proportions.
- Content volume by release year / type.
- Genres/countries/ratings where data is present.
- Duration patterns (minutes vs seasons — NOT directly comparable).
- Catalog release-year distribution.

In [1]:
import matplotlib
matplotlib.use("Agg")
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

def resolve_repo_root():
    cwd = Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if (parent / "Dataset").is_dir() and (parent / "projects").is_dir():
            return parent
        if (parent / "Dataset").is_dir():
            return parent
    return Path(__file__).resolve().parents[3]

REPO_ROOT = resolve_repo_root()
DATASET_DIR = REPO_ROOT / "Dataset"
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
def fig_path(name):
    return str(OUT_DIR / name)

print("Repo root:", REPO_ROOT)
print("Dataset dir:", DATASET_DIR)
print("Output dir:", OUT_DIR)

SRC_FILE = DATASET_DIR / "netflix_titles.csv"
print("Source file:", SRC_FILE)
assert SRC_FILE.exists(), f"Source file missing: {SRC_FILE}"
df = pd.read_csv(SRC_FILE, encoding="utf-8")
print("Loaded shape:", df.shape)


Repo root: C:\Users\sohan\OneDrive\Desktop\CyberNova
Dataset dir: C:\Users\sohan\OneDrive\Desktop\CyberNova\Dataset
Output dir: outputs
Source file: C:\Users\sohan\OneDrive\Desktop\CyberNova\Dataset\netflix_titles.csv
Loaded shape: (8807, 12)


In [2]:
print("dimensions:", df.shape)
print("columns:", list(df.columns))
print("dtypes:"); print(df.dtypes.to_string())
print("sample:"); print(df.head(3).to_string())
print("null counts:"); print(df.isnull().sum().to_string())
print("duplicates:", int(df.duplicated().sum()))
print("type:", df["type"].value_counts().to_dict())

dimensions: (8807, 12)
columns: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']
dtypes:
show_id           str
type              str
title             str
director          str
cast              str
country           str
date_added        str
release_year    int64
rating            str
duration          str
listed_in         str
description       str
sample:
  show_id     type                 title         director                                                                                                                                                                                                                                                                                                             cast        country          date_added  release_year rating   duration                                                      listed_in                                                        

In [3]:
print("null counts per column:")
print(df.isnull().sum().to_string())
print("missing director:", int(df["director"].isnull().sum()), "| cast:", int(df["cast"].isnull().sum()))
print("missing country:", int(df["country"].isnull().sum()), "| rating:", int(df["rating"].isnull().sum()))
print("missing date_added:", int(df["date_added"].isnull().sum()), "| duration:", int(df["duration"].isnull().sum()))
print("rating unique:"); print(df["rating"].value_counts().to_dict())
print("duration top:"); print(df["duration"].value_counts().head(12).to_dict())

null counts per column:
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
missing director: 2634 | cast: 825
missing country: 831 | rating: 4
missing date_added: 10 | duration: 3
rating unique:
{'TV-MA': 3207, 'TV-14': 2160, 'TV-PG': 863, 'R': 799, 'PG-13': 490, 'TV-Y7': 334, 'TV-Y': 307, 'PG': 287, 'TV-G': 220, 'NR': 80, 'G': 41, 'TV-Y7-FV': 6, 'NC-17': 3, 'UR': 3, '74 min': 1, '84 min': 1, '66 min': 1}
duration top:
{'1 Season': 1793, '2 Seasons': 425, '3 Seasons': 199, '90 min': 152, '94 min': 146, '97 min': 146, '93 min': 146, '91 min': 144, '95 min': 137, '96 min': 130, '92 min': 129, '102 min': 122}


In [4]:
clean = df.copy()
clean["country_clean"] = clean["country"].astype(str).str.strip()
clean["listed_in_clean"] = clean["listed_in"].astype(str).str.strip()
clean["director_clean"] = clean["director"].fillna("Not listed")
clean["cast_clean"] = clean["cast"].fillna("Not listed")
clean["rating_clean"] = clean["rating"].fillna("Not rated")

def parse_duration(x):
    if pd.isna(x): return (np.nan, np.nan)
    s = str(x).strip()
    if s.endswith(" min"):
        try: return (float(s.replace(" min","")), "minutes")
        except Exception: return (np.nan, np.nan)
    if s.endswith(" Seasons") or s.endswith(" Season"):
        try: return (float(s.replace(" Seasons","").replace(" Season","")), "seasons")
        except Exception: return (np.nan, np.nan)
    return (np.nan, np.nan)

dur = clean["duration"].apply(parse_duration)
clean["duration_num"] = [d[0] for d in dur]
clean["duration_unit"] = [d[1] for d in dur]
print("cleaned rows:", len(clean))
print("duration unit counts:", clean["duration_unit"].value_counts(dropna=False).to_dict())
print("movie minutes:", int((clean["duration_unit"]=="minutes").sum()), "| tv seasons:", int((clean["duration_unit"]=="seasons").sum()))
clean.to_csv(OUT_DIR / "cleaned_streaming_content.csv", index=False)
print("saved -> outputs/cleaned_streaming_content.csv")

cleaned rows: 8807
duration unit counts: {'minutes': 6128, 'seasons': 2676, nan: 3}
movie minutes: 6128 | tv seasons: 2676


saved -> outputs/cleaned_streaming_content.csv


In [5]:
print("type split:"); print(df["type"].value_counts().to_string())
print("proportions:"); print((df["type"].value_counts(normalize=True)*100).round(1).to_string())
print("release year range:", df["release_year"].min(), "-", df["release_year"].max())
print("distinct release years:", df["release_year"].nunique())
print("recent 12 years:"); print(df["release_year"].value_counts().sort_index().tail(12).to_string())
print("rating:"); print(df["rating"].value_counts().to_string())
countries = df["country"].dropna().astype(str).str.split(",").explode().str.strip()
genres = df["listed_in"].dropna().astype(str).str.split(",").explode().str.strip()
print("top countries:"); print(countries.value_counts().head(10).to_string())
print("top genres:"); print(genres.value_counts().head(10).to_string())
min_d = df.loc[df["duration"].str.contains(" min", na=False), "duration"].str.replace(" min","").astype(float)
sea_d = df.loc[df["duration"].str.contains("Season", na=False), "duration"].str.replace(" Seasons","").str.replace(" Season","").astype(float)
print("movie median minutes:", min_d.median(), "| season median:", sea_d.median())

type split:
type
Movie      6131
TV Show    2676
proportions:
type
Movie      69.6
TV Show    30.4
release year range: 1925 - 2021
distinct release years: 74
recent 12 years:
release_year
2010     194
2011     185
2012     237
2013     288
2014     352
2015     560
2016     902
2017    1032
2018    1147
2019    1030
2020     953
2021     592
rating:


rating
TV-MA       3207
TV-14       2160
TV-PG        863
R            799
PG-13        490
TV-Y7        334
TV-Y         307
PG           287
TV-G         220
NR            80
G             41
TV-Y7-FV       6
NC-17          3
UR             3
74 min         1
84 min         1
66 min         1
top countries:
country
United States     3690
India             1046
United Kingdom     806
Canada             445
France             393
Japan              318
Spain              232
South Korea        231
Germany            226
Mexico             169
top genres:
listed_in
International Movies        2752
Dramas                      2427
Comedies                    1674
International TV Shows      1351
Documentaries                869
Action & Adventure           859
TV Dramas                    763
Independent Movies           756
Children & Family Movies     641
Romantic Movies              616


movie median minutes: 98.0 | season median: 1.0


In [6]:
import os
os.makedirs(OUT_DIR, exist_ok=True)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
df["type"].value_counts().plot(kind="bar", ax=axes[0,0], color=["#1f77b4","#ff7f0e"])
axes[0,0].set_title("Title Type Split"); axes[0,0].set_ylabel("Count"); axes[0,0].tick_params(axis="x", rotation=0)
rel = df["release_year"].value_counts().sort_index()
rel.plot(kind="line", marker="o", ax=axes[0,1], color="#2ca02c")
axes[0,1].set_title("Releases by Release Year"); axes[0,1].set_xlabel("Year"); axes[0,1].set_ylabel("Count")
genres.value_counts().head(10).plot(kind="barh", ax=axes[0,2], color="#9467bd")
axes[0,2].set_title("Top Genres"); axes[0,2].set_xlabel("Count")
df["rating"].value_counts().head(10).plot(kind="bar", ax=axes[1,0], color="#d62728")
axes[1,0].set_title("Rating Distribution"); axes[1,0].tick_params(axis="x", rotation=45); axes[1,0].set_ylabel("Count")
countries.value_counts().head(10).plot(kind="barh", ax=axes[1,1], color="#8c564b")
axes[1,1].set_title("Top Countries"); axes[1,1].set_xlabel("Count")
min_d.plot(kind="hist", bins=20, ax=axes[1,2], color="#e377c2", edgecolor="white")
axes[1,2].set_title("Movie Duration (min)"); axes[1,2].set_xlabel("Minutes")
fig.tight_layout(); fig.savefig(fig_path("charts_streaming.png"), dpi=150, bbox_inches="tight"); plt.close(fig)
print("saved outputs/charts_streaming.png")

saved outputs/charts_streaming.png


In [7]:
print("KEY FINDINGS (streaming):")
print("1. Movies %d (%.1f%%) | TV Shows %d (%.1f%%)" % (
    (df.type=="Movie").sum(), (df.type=="Movie").mean()*100,
    (df.type=="TV Show").sum(), (df.type=="TV Show").mean()*100))
print("2. Release years %d to %d (%d distinct)" % (df["release_year"].min(), df["release_year"].max(), df["release_year"].nunique()))
print("3. Most common rating: %s (%d titles); R-rated: %d" % (
    df.rating.value_counts().idxmax(), df.rating.value_counts().iloc[0], (df.rating=="R").sum()))
print("4. Top country: %s (%d); top genre: %s (%d)" % (
    countries.value_counts().idxmax(), countries.value_counts().iloc[0],
    genres.value_counts().idxmax(), genres.value_counts().iloc[0]))
print("5. Median movie runtime %.1f min; median TV %.1f seasons (NOT comparable units)" % (min_d.median(), sea_d.median()))
print("6. Director missing %d (%.1f%%); country missing %d -- treated as missing, not invented" % (
    df.director.isnull().sum(), df.director.isnull().mean()*100, df.country.isnull().sum()))
print("RECOMMENDATIONS:")
print("- Align acquisitions with top ratings/countries present in catalog.")
print("- Keep movie minutes and TV seasons on separate metrics; never compare directly.")
print("- Investigate missing director/country fields if completeness is required.")

KEY FINDINGS (streaming):
1. Movies 6131 (69.6%) | TV Shows 2676 (30.4%)
2. Release years 1925 to 2021 (74 distinct)
3. Most common rating: TV-MA (3207 titles); R-rated: 799
4. Top country: United States (3690); top genre: International Movies (2752)
5. Median movie runtime 98.0 min; median TV 1.0 seasons (NOT comparable units)
6. Director missing 2634 (29.9%); country missing 831 -- treated as missing, not invented
RECOMMENDATIONS:
- Align acquisitions with top ratings/countries present in catalog.
- Keep movie minutes and TV seasons on separate metrics; never compare directly.
- Investigate missing director/country fields if completeness is required.


In [8]:
print("LIMITATIONS:")
print("- Director/cast/country/rating missing values treated as missing, never imputed with invented values.")
print("- Comma-separated country/genre -> title may count in multiple categories.")
print("- Duration mixes 'min' and 'Seasons' -> not directly comparable.")
print("- Catalog snapshot (historical data), not live catalog.")
print("CONCLUSION: The catalog is movie-majority (~69.6%%), with US and India dominant. Missing descriptive fields "
      "are handled explicitly rather than imputed, and movie minutes vs TV seasons are kept on separate scales.")

LIMITATIONS:
- Director/cast/country/rating missing values treated as missing, never imputed with invented values.
- Comma-separated country/genre -> title may count in multiple categories.
- Duration mixes 'min' and 'Seasons' -> not directly comparable.
- Catalog snapshot (historical data), not live catalog.
CONCLUSION: The catalog is movie-majority (~69.6%%), with US and India dominant. Missing descriptive fields are handled explicitly rather than imputed, and movie minutes vs TV seasons are kept on separate scales.
